# End-to-end reproduction: posterior bias comparison



# 0. Environment initialisation

Install the packages required by this notebook.

In [ ]:
%matplotlib inline

# Standard library
from pathlib import Path
import json
import pickle
import sys

# Numerical computing
import jax
import jax.numpy as jnp
import numpy as np
import torch

# Plotting
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde

Locate the project root and make the project modules importable in the current
notebook kernel.

In [2]:
# Locate the project root.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "Example" else cwd

# Make the project modules importable in this notebook.
project_paths = [
    PROJECT_ROOT,
    PROJECT_ROOT / "priorg",
]

for path in project_paths:
    path_str = str(path)

    if path_str not in sys.path:
        sys.path.insert(0, path_str)

print("Project root:", PROJECT_ROOT)


Import the functions required for task construction, adaptation, posterior sampling, and plotting.

In [3]:
# Task construction
from priorg.sim.tasks.task import get_task

# Data augmentation
from priorg.sim.methods.data_augment import (
    build_augmented_dataset,
)

# KLIEP guidance
from priorg.sim.methods.guidance_dre import (
    init_log_ratio_model,
    train_log_ratio_model_kliep,
    prior_guide_with_log_ratio_model,
    sample_unguided,
)

# PriorGuide GMM fitting
from priorg.sim.methods.gmm import (
    fit_gmm_v2,
)

# PriorGuide sampling
from priorg.sim.methods.guidance_priorguide import (
    prior_guide_theta_prior_only,
)

# Standardisation helpers used by the original experiment script
from experiments.posterior.run_guidance import (
    _un_z_score_latent_if_needed,
    _z_score_gaussian_mixture,
    _z_score_observation_if_needed,
)

# 1. Experiment configuration

Set the task, observation seed, and experimental hyperparameters used for reproduction.


In [4]:
# Directory paths
EXAMPLE_DIR = PROJECT_ROOT / "Example"
DATA_DIR = EXAMPLE_DIR / "Data"
MODEL_DIR = EXAMPLE_DIR / "Model"

# Output path
RESULTS_PATH = PROJECT_ROOT / "results"

# Task configuration
TASK_NAME = "oup"
TASK_TYPE = "id"
MODEL_ID = 0
OBSERVATION_SEED = 1000003
DIMS = (0, 1)

# KLIEP guidance training
GUIDANCE_TRAIN_EPOCHS = 10000
GUIDANCE_TRAIN_BATCH_SIZE = 4096
GUIDANCE_TRAIN_LR = 3e-4

# Data augmentation
AUGMENT_SAMPLES = 420000
GUIDANCE_SEED = 0

# PriorGuide GMM fitting
PRIORGUIDE_NUM_COMPONENTS = 50
PRIORGUIDE_NUM_ITERATIONS = 10000
PRIORGUIDE_LEARNING_RATE = 1e-3
PRIORGUIDE_BATCH_SIZE = 200

# Posterior sampling
NUM_SAMPLES = 5000
NUM_DIFFUSION_STEPS = 25
NUM_BATCHES = 5
RHO = 2.0
GUIDANCE_SCALE = 1.0
PRIORGUIDE_LANGEVIN_STEPS = 4
PRIORGUIDE_LANGEVIN_RATIO = 0.5

# Plotting
CONTOUR_LEVELS = 4



# 2. Load reference data

Locate the provided observation file and load the reference posterior samples
used in the final OUP figure.

In [5]:
observation_file = DATA_DIR / f"obs_{OBSERVATION_SEED}.json"
reference_file = DATA_DIR / f"ref_{OBSERVATION_SEED}.npz"

# Load the reference posterior samples.
with np.load(reference_file) as reference_data:
    reference_samples = reference_data["samples"]

print("Reference samples:", reference_samples.shape)


Reference samples: (10000, 2)


# 3. Load the pretrained base model

Load the pretrained diffusion score model used in the OUP in-distribution experiment.

The model was trained with the following configuration:

- Task: `oup_id`
- Training simulations: `50,000`
- Training steps: `50,000`
- Batch size: `1,000`
- Learning rate: `1e-3`
- SDE: `VPSDE`
- Random seed: `0`

In [6]:
# Load the pretrained base diffusion score model.
base_model_file = (
    MODEL_DIR
    / f"model_{MODEL_ID}.pkl"
)

with base_model_file.open("rb") as f:
    base_model = pickle.load(f)

print("Loaded base model:", base_model_file)

# 4. Prepare the adaptation methods

## 4.1 Our mothod：Train the KLIEP guidance model

Build the OUP task, augment the original base-model training dataset with
model-generated parameter–observation pairs, and train a KLIEP log-ratio model.

Build the OUP task and prepare the augmented training dataset. The cached
augmented dataset (`augmented_data_0_420000.pkl`) is loaded when present;
otherwise it is regenerated from `data_0.pkl` and cached.

In [7]:
# Build the task with the reference and target priors.
task = get_task(f"{TASK_NAME}_{TASK_TYPE}")

# Reuse the cached augmented dataset when available; otherwise build it
# from the base training dataset, exactly as in run_guidance.py.
augmented_cache_file = (
    DATA_DIR
    / f"augmented_data_{MODEL_ID}_{AUGMENT_SAMPLES}.pkl"
)

if augmented_cache_file.exists():
    with augmented_cache_file.open("rb") as f:
        guidance_dataset = pickle.load(f)

    print("Loaded cached augmented dataset:", augmented_cache_file)
else:
    # Load the original base-model training dataset.
    base_data_file = DATA_DIR / f"data_{MODEL_ID}.pkl"

    with base_data_file.open("rb") as f:
        base_dataset = pickle.load(f)

    # Number of latent parameter dimensions.
    theta_dim = int(
        (~base_model.condition_mask.astype(bool)).sum()
    )

    # Match the augmentation seed used in run_guidance.py.
    augmentation_key = jax.random.PRNGKey(
        GUIDANCE_SEED + 99999
    )

    # Generate the augmented training dataset.
    guidance_dataset = build_augmented_dataset(
        base_dataset=base_dataset,
        model=base_model,
        condition_mask=base_model.condition_mask,
        theta_dim=theta_dim,
        n_augment=AUGMENT_SAMPLES,
        key=augmentation_key,
        num_steps=NUM_DIFFUSION_STEPS,
        rho=RHO,
    )

    # Cache for future runs.
    with augmented_cache_file.open("wb") as f:
        pickle.dump(guidance_dataset, f)

    print("Augmented dataset cached to:", augmented_cache_file)

print(
    "Augmented guidance dataset:",
    f"theta={guidance_dataset['theta'].shape},",
    f"x={guidance_dataset['x'].shape}",
)

Train the KLIEP guidance model. 

In [8]:
# Initialise the random key for KLIEP training.
key = jax.random.PRNGKey(GUIDANCE_SEED)
key, key_init = jax.random.split(key)

# Initialise the KLIEP log-ratio MLP.
kliep_spec = init_log_ratio_model(
    base_model,
    key_init,
    reinit_output_head=True,
    model_type="mlp",
    mlp_hidden_dim=256,
    mlp_num_layers=4,
    mlp_time_embedding_dim=4,
    mlp_activation="silu",
)

# Split a separate key for training.
key, key_train = jax.random.split(key)

# Train the KLIEP guidance model.
kliep_spec, kliep_loss_history = train_log_ratio_model_kliep(
    task=task,
    diffusion_model=base_model,
    condition_mask=base_model.condition_mask,
    model_spec=kliep_spec,
    rng=key_train,
    batch_size=GUIDANCE_TRAIN_BATCH_SIZE,
    epochs=GUIDANCE_TRAIN_EPOCHS,
    lr=GUIDANCE_TRAIN_LR,
    patience=0,
    dataset=guidance_dataset,
)

print(
    "KLIEP training complete | "
    f"epochs={len(kliep_loss_history)} | "
    f"final loss={float(kliep_loss_history[-1]):.4f}"
)

KLIEP training complete | epochs=10000 | final loss=-0.1685


## 4.2 PriorGuide：Fit the Gaussian mixture model

Fit the Gaussian mixture model required by the PriorGuide baseline.The model approximates the target-to-reference prior log-ratio using samples from the target prior. The fitted mixture parameters are transformed into the
standardised parameter space used by the pretrained diffusion model

Fit the Gaussian mixture model


In [9]:
# Obtain the target and reference priors.
target_prior = task.get_target_prior()
reference_prior = task.get_reference_prior()

# Define the target-to-reference prior log-ratio.
def prior_log_ratio(theta):
    return (
        target_prior.log_prob(theta)
        - reference_prior.log_prob(theta)
    )

# Draw training samples from the target prior.
def sample_target_prior(batch_size):
    return target_prior.sample((batch_size,))

# Fit the PriorGuide Gaussian mixture model.
priorguide_gmm = fit_gmm_v2(
    log_target_fn=prior_log_ratio,
    data_sampler=sample_target_prior,
    num_components=PRIORGUIDE_NUM_COMPONENTS,
    dim=task.theta_dim,
    num_iters=PRIORGUIDE_NUM_ITERATIONS,
    learning_rate=PRIORGUIDE_LEARNING_RATE,
    batch_size=PRIORGUIDE_BATCH_SIZE,
)

print(
    "PriorGuide GMM fitted:",
    f"{PRIORGUIDE_NUM_COMPONENTS} components",
)

fit_gmm_v2 iter 0, loss: 7.7443
fit_gmm_v2 iter 1000, loss: 1.1626
fit_gmm_v2 iter 2000, loss: 0.1833
fit_gmm_v2 iter 3000, loss: 0.0138
fit_gmm_v2 iter 4000, loss: 0.0021
fit_gmm_v2 iter 5000, loss: 0.0004
fit_gmm_v2 iter 6000, loss: 0.0001
fit_gmm_v2 iter 7000, loss: 0.0000
fit_gmm_v2 iter 8000, loss: 0.0000
fit_gmm_v2 iter 9000, loss: 0.0000
PriorGuide GMM fitted: 50 components


Transform the mixture parameters for posterior sampling

In [10]:

# Extract the fitted mixture parameters.
gmm_log_weights = jnp.asarray(
    priorguide_gmm.log_weights.detach().cpu().numpy()
)

gmm_means = jnp.asarray(
    priorguide_gmm.means.detach().cpu().numpy()
)

gmm_stds = jnp.asarray(
    (
        torch.exp(priorguide_gmm.log_stds)
        + 1e-6
    ).detach().cpu().numpy()
)

# Construct diagonal covariance matrices.
gmm_covs = (
    gmm_stds**2
)[..., None] * jnp.eye(gmm_stds.shape[-1])


# Identify the latent parameter nodes.
latent_mask = ~base_model.condition_mask.astype(bool)

latent_node_id = base_model.node_id[
    latent_mask
]

# Transform the mixture into the model's standardised space.
(
    target_log_weights_z,
    target_means_z,
    target_covs_z,
) = _z_score_gaussian_mixture(
    base_model,
    gmm_log_weights,
    gmm_means,
    gmm_covs,
    latent_node_id,
)

print(
    "PriorGuide mixture parameters:",
    f"weights={target_log_weights_z.shape},",
    f"means={target_means_z.shape},",
    f"covariances={target_covs_z.shape}",
)

PriorGuide mixture parameters: weights=(50,), means=(50, 2), covariances=(50, 2, 2)


# 5. Generate posterior samples

Generate posterior samples for the selected observation using:

- **Simformer:** the pretrained base model without prior adaptation;
- **KLIEP guidance:** the base model combined with the trained KLIEP log-ratio model;
- **PriorGuide:** the base model combined with the fitted Gaussian mixture prior.

Sampling is performed in five batches, and the results are saved for the
headline posterior comparison figure.

## 5.1 Load the observation

Load the selected parameter–observation pair and prepare the standardised observation required by PriorGuide.

In [11]:
# Load the selected observation.
with observation_file.open("r") as f:
    observation = json.load(f)

theta_o = jnp.atleast_1d(
    jnp.asarray(observation["theta"])
)

x_o = jnp.atleast_1d(
    jnp.asarray(observation["x"])
)

# PriorGuide operates on the standardised observation.
x_o_z = _z_score_observation_if_needed(
    base_model,
    x_o,
)

theta_dim = task.theta_dim
x_dim = task.x_dim

print("Observation seed:", OBSERVATION_SEED)
print("True parameter:", theta_o)
print("Observation shape:", x_o.shape)

Observation seed: 1000003
True parameter: [ 2.176 -1.441]
Observation shape: (25,)


## 5.2 Define the posterior samplers

Define the single-sample functions used by KLIEP guidance and PriorGuide.

In [12]:
def draw_terminal_state(key):
    """Draw the terminal state of the reverse diffusion process."""
    state_dim = theta_dim + x_dim
    zeros = jnp.zeros((state_dim,))

    return (
        base_model.sde.marginal_mean(
            jnp.array([base_model.T_max]),
            zeros,
        ).reshape(-1)
        + base_model.sde.marginal_stddev(
            jnp.array([base_model.T_max]),
            zeros,
        ).reshape(-1)
        * jax.random.normal(
            key,
            shape=(state_dim,),
        )
    )

    
@jax.jit
def sample_once_simformer(key):
    """Generate one unguided base-model sample, with the same sampler as the guided methods."""
    return sample_unguided(
        model=base_model,
        key=key,
        condition_mask=base_model.condition_mask,
        x_o=x_o,
        x_T=draw_terminal_state(key),
        num_steps=NUM_DIFFUSION_STEPS,
        rho=RHO,
    )


@jax.jit
def sample_once_kliep(key):
    """Generate one KLIEP-guided posterior sample."""
    return prior_guide_with_log_ratio_model(
        model=base_model,
        key=key,
        condition_mask=base_model.condition_mask,
        x_o=x_o,
        x_T=draw_terminal_state(key),
        log_ratio_apply=kliep_spec.log_ratio_apply,
        log_ratio_params=kliep_spec.params,
        num_steps=NUM_DIFFUSION_STEPS,
        rho=RHO,
        guidance_scale=GUIDANCE_SCALE,
    )


@jax.jit
def sample_once_priorguide(key):
    """Generate one PriorGuide posterior sample."""
    result = prior_guide_theta_prior_only(
        model=base_model,
        key=key,
        condition_mask=base_model.condition_mask,
        x_o=x_o_z,
        x_T=draw_terminal_state(key),
        theta_prior_log_weights=target_log_weights_z,
        theta_prior_mean=target_means_z,
        theta_prior_cov=target_covs_z,
        prior_dim=theta_dim,
        num_steps=NUM_DIFFUSION_STEPS,
        rho=RHO,
        langevin_steps=PRIORGUIDE_LANGEVIN_STEPS,
        langevin_ratio=PRIORGUIDE_LANGEVIN_RATIO,
    )

    return _un_z_score_latent_if_needed(
        base_model,
        result,
    )

## 5.3 Generate and save posterior samples

In [13]:
if NUM_SAMPLES % NUM_BATCHES != 0:
    raise ValueError(
        "NUM_SAMPLES must be divisible by NUM_BATCHES."
    )

batch_size = NUM_SAMPLES // NUM_BATCHES

simformer_batches = []
kliep_batches = []
priorguide_batches = []

# Reconstruct the sampling key used after KLIEP initialisation and training.
key = jax.random.PRNGKey(GUIDANCE_SEED)
key, _ = jax.random.split(key)  # KLIEP initialisation
key, _ = jax.random.split(key)  # KLIEP training
key, key_obs = jax.random.split(key)

for batch_idx in range(NUM_BATCHES):
    print(f"Batch {batch_idx + 1}/{NUM_BATCHES}")

    # Match the key allocation used in run_guidance.py.
    batch_keys = jax.random.split(key_obs, 11)

    key_obs = batch_keys[0]
    key_simformer = batch_keys[2]
    key_priorguide = batch_keys[3]
    key_kliep = batch_keys[6]

    
    # Simformer
    simformer_batch = jax.vmap(
        sample_once_simformer
    )(
        jax.random.split(key_simformer, batch_size)
    )[:, :theta_dim]
    
    
    # KLIEP guidance
    kliep_batch = jax.vmap(
        sample_once_kliep
    )(
        jax.random.split(key_kliep, batch_size)
    )[:, :theta_dim]

    
    # PriorGuide
    priorguide_batch = jax.vmap(
        sample_once_priorguide
    )(
        jax.random.split(key_priorguide, batch_size)
    )[:, :theta_dim]

    simformer_batches.append(simformer_batch)
    kliep_batches.append(kliep_batch)
    priorguide_batches.append(priorguide_batch)


# Concatenate the five batches.
simformer_samples = jnp.concatenate(
    simformer_batches,
    axis=0,
)

kliep_samples = jnp.concatenate(
    kliep_batches,
    axis=0,
)

priorguide_samples = jnp.concatenate(
    priorguide_batches,
    axis=0,
)


samples_to_save = {
    "simformer": simformer_samples,
    "kliep_guidance": kliep_samples,
    "priorguide": priorguide_samples,
}

for method, samples in samples_to_save.items():
    output_dir = (
        RESULTS_PATH
        / method
        / TASK_NAME
        / TASK_TYPE
        / f"model_{MODEL_ID}"
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = (
        output_dir
        / (
            f"samples_{NUM_SAMPLES}_"
            f"obs_{OBSERVATION_SEED}_"
            "run_0.npz"
        )
    )

    np.savez(
        output_file,
        samples=np.asarray(samples),
        theta=np.asarray(theta_o),
        x=np.asarray(x_o),
    )

    print("Saved:", output_file)

Batch 1/5


# 6. Plot the posterior comparison

Use the generated posterior samples to reproduce the headline OUP posterior
comparison.

In [14]:
def kde_grid(samples, dx, dy, pad=0.12, grid=100):
    xs, ys = samples[:, dx], samples[:, dy]

    xp = pad * (xs.max() - xs.min())
    yp = pad * (ys.max() - ys.min())

    xx, yy = np.mgrid[
        xs.min() - xp : xs.max() + xp : complex(grid),
        ys.min() - yp : ys.max() + yp : complex(grid),
    ]

    kde = gaussian_kde(
        np.vstack([xs, ys])
    )

    zz = kde(
        np.vstack([xx.ravel(), yy.ravel()])
    ).reshape(xx.shape)

    return xx, yy, zz / zz.max()

In [ ]:
dx, dy = DIMS
levels = np.linspace(0.05, 1.0, CONTOUR_LEVELS + 1)

fig, ax = plt.subplots(figsize=(4.5, 4.2))

# Reference posterior
xx, yy, zz = kde_grid(
    np.asarray(reference_samples),
    dx,
    dy,
    pad=0.04,
)

ax.contourf(
    xx,
    yy,
    zz,
    levels=levels,
    cmap="Greys",
    alpha=0.6,
    zorder=1,
    vmin=-0.3,
    vmax=1.0,
)

# Posterior samples from the three methods
method_specs = [
    (
        simformer_samples,
        "Before Guidance",
        "#666666",
        "dashed",
    ),
    (
        kliep_samples,
        "After Guidance (Ours)",
        "#D55E00",
        "solid",
    ),
    (
        priorguide_samples,
        "After Guidance (PriorGuide)",
        "#009E73",
        "solid",
    ),
]

for samples, label, color, linestyle in method_specs:
    xx_method, yy_method, zz_method = kde_grid(
        np.asarray(samples),
        dx,
        dy,
    )

    ax.contour(
        xx_method,
        yy_method,
        zz_method,
        levels=levels,
        colors=color,
        linewidths=2.2,
        linestyles=linestyle,
        zorder=2,
    )

# True parameter
theta_true = np.asarray(theta_o)

ax.scatter(
    theta_true[dx],
    theta_true[dy],
    marker="*",
    s=200,
    color="black",
    zorder=5,
    clip_on=False,
)

# Axes
ax.set_xlim(xx.min(), xx.max())
ax.set_ylim(-3.0, -0.4)

ax.set_xlabel(
    f"$\\theta_{dx + 1}$",
    fontsize=13,
)
ax.set_ylabel(
    f"$\\theta_{dy + 1}$",
    fontsize=13,
)
ax.tick_params(labelsize=10)

# Legend
legend_handles = [
    mpatches.Patch(
        facecolor="grey",
        alpha=0.6,
        label="True Posterior",
    ),
    plt.Line2D(
        [],
        [],
        marker="*",
        color="black",
        linestyle="None",
        markersize=11,
        label="True parameter",
    ),
    plt.Line2D([], [], color="none", label=" "),
]

legend_handles.extend(
    plt.Line2D(
        [],
        [],
        color=color,
        linewidth=2.2,
        linestyle=linestyle,
        label=label,
    )
    for _, label, color, linestyle in method_specs
)

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    fontsize=11,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)

fig.tight_layout(rect=[0, 0.18, 1, 1])

# Save the headline figure
plot_dir = RESULTS_PATH / "plots" / "paper"
plot_dir.mkdir(parents=True, exist_ok=True)

pdf_path = plot_dir / "posterior_bias_comparison.pdf"
png_path = plot_dir / "posterior_bias_comparison.png"

fig.savefig(
    pdf_path,
    dpi=200,
    bbox_inches="tight",
)
fig.savefig(
    png_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print("Saved:", pdf_path)
print("Saved:", png_path)